# IT135IU Group 3, Topic 8: D3 Analysis
**Decision question:** which film genres should the Content Marketing and Promotion Team promote to each gender x age-group segment, instead of promoting the same genres to everyone? **Data:** MovieLens 1M (GroupLens).

| No. | Role | Name | Student ID |
|-----|------|------|------------|
| 1 | Product owner | Truc Phuong | ITDSIU25031 |
| 2 | Data engineer | Le Quan | ITDSIU25033 |
| 3 | Data analyst A | Que | ITDSIU25036 |
| 4 | Data analyst B | Hoang Quan | ITDSIU25035 |
| 5 | Visualisation lead | Tuyen | ITDSIU23028 |
| 6 | Reproducibility lead | Trung | ITDSIU24054 |
| 7 | Ethics and privacy officer | Cuong | ITDSIU25008 |

## Source and terms of use
- **Dataset:** MovieLens 1M, [grouplens.org/datasets/movielens/1m/](https://grouplens.org/datasets/movielens/1m/) (file loaded in Step 1 from [files.grouplens.org/datasets/movielens/ml-1m.zip](https://files.grouplens.org/datasets/movielens/ml-1m.zip))
- **Publisher:** GroupLens Research, University of Minnesota
- **Date of retrieval:** 28/09/2026
- **Terms of use** (copied from the README in ml-1m.zip):
  Neither the University of Minnesota nor any of the researchers involved can guarantee the correctness of the data, its suitability for any particular purpose, or the validity of results based on the use of the data set. The data set may be used for any research purposes under the following conditions:
  - The user may not state or imply any endorsement from the University of Minnesota or the GroupLens Research Group.
  - The user must acknowledge the use of the data set in publications resulting from the use of the data set (see below for citation information).
  - The user may not redistribute the data without separate permission.
  - The user may not use this information for any commercial or revenue-bearing purposes without first obtaining permission from a faculty member of the GroupLens Research Project at the University of Minnesota.

**Citation:**
> F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. *ACM Transactions on Interactive Intelligent Systems (TiiS)* 5, 4, Article 19 (December 2015), 19 pages. DOI: [http://dx.doi.org/10.1145/2827872](http://dx.doi.org/10.1145/2827872)

## Terms used in this notebook
- **segment**: one gender × age group, e.g. F 25-34 (2 × 7 = 14 segments).
- **cell**: one segment × genre pair, e.g. F 25-34 × Drama (14 × 18 = 252 cells). Code blocks are always referred to by their Step number, never as cells.
- **tier**: reliability label of a cell, based on its number of distinct users.
- **satisfaction rate**: share of a cell's ratings that are 4 or 5 stars.

## Terms used in this notebook
- **Segment:** one gender × age group (2 × 7 = 14 segments).
- **Cell:** one segment × genre pair (14 × 18 = 252 cells).
- **Step:** one code block of this notebook. Steps 1 to 9 are carried over unchanged from D2. D3 Steps are named after the analysis steps: Step B1 to Step B9 (analysis), Step B10.1 to Step B10.4 (charts), Step Q1 and Step Q2 (descriptive and summary tables).
- **Satisfaction rate:** share of a cell's ratings that are 4 or 5 stars (rating-level).
- **Tier:** a cell is *reliable* with at least 100 distinct users, *indicative* otherwise. Tiers are computed once on the full data (Step 7) and kept on both halves.
- **Discovery half / validation half:** users of each segment are split at random into two halves (Step B1). Genres are chosen on the discovery half and tested on the validation half.

### Step 1: Load the Three Raw Tables
Load the raw files (`ratings.dat`, `users.dat`, and `movies.dat`) from the MovieLens zip file. The file is downloaded from GroupLens; if the download fails three times, a local copy in `data/ml-1m.zip` is used (cleaning log C19).

In [1]:
import io
import ssl
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import re

SEED = 42
np.random.seed(SEED)

URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
LOCAL_ZIP = Path("data/ml-1m.zip")  # used only if the download fails; never committed
READ: dict[str, Any] = {
    "sep": "::",
    "engine": "python",
    "encoding": "latin-1",
    "header": None,
}

AGE_LABEL = {
    1: "Under 18",
    18: "18-24",
    25: "25-34",
    35: "35-44",
    45: "45-49",
    50: "50-55",
    56: "56+",
}

def load(archive: zipfile.ZipFile, name: str, cols: list[str]) -> pd.DataFrame:
    """Read one "::" separated member straight out of the archive."""
    data = io.BytesIO(archive.read(f"ml-1m/{name}"))
    return pd.read_csv(data, names=cols, **READ)


def fetch_zip(url: str, local: Path, tries: int = 3, wait: int = 5) -> zipfile.ZipFile:
    """Download the archive with certifi certificates; fall back to a local copy.

    Certificate verification is never turned off. If every attempt fails and
    there is no local copy, stop with instructions for a manual download.
    """
    try:
        import certifi
        context = ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        context = ssl.create_default_context()

    last_error = None
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(url, context=context, timeout=60) as resp:
                data = resp.read()
            print(f"source : {url} (download, attempt {attempt})")
            return zipfile.ZipFile(io.BytesIO(data))
        except Exception as err:  # network or certificate error
            last_error = err
            print(f"download attempt {attempt} failed: {type(err).__name__}: {err}")
            if attempt < tries:
                time.sleep(wait)

    if local.exists():
        print(f"source : local copy {local}")
        return zipfile.ZipFile(local)

    raise RuntimeError(
        f"Could not download {url} ({last_error}) and found no local copy at {local}. "
        "Download ml-1m.zip by hand from https://grouplens.org/datasets/movielens/1m/ "
        "and put it in a folder named data/ next to this notebook."
    )

print("STEP 1  load the three raw tables")
zf = fetch_zip(URL, LOCAL_ZIP)
print("members inside zip:")
for member in zf.namelist():
    print(f"  - {member}")

ratings = load(zf, "ratings.dat", ["user_id", "movie_id", "rating", "timestamp"])
users = load(zf, "users.dat", ["user_id", "gender", "age", "occupation", "zip"])
movies = load(zf, "movies.dat", ["movie_id", "title", "genres"])

STEP 1  load the three raw tables
source : https://files.grouplens.org/datasets/movielens/ml-1m.zip (download, attempt 1)
members inside zip:
  - ml-1m/
  - ml-1m/movies.dat
  - ml-1m/ratings.dat
  - ml-1m/README
  - ml-1m/users.dat


### Step 2: Missing Values, Keys, and Orphan Rows
Validate the integrity of our tables by checking for missing values, confirming primary keys are unique, and identifying any orphaned foreign keys.

In [2]:
print("STEP 2  missing values, keys and orphan rows")

for name, table in [
    ("ratings", ratings),
    ("users", users),
    ("movies", movies),
]:
    print(f"\n{name}: shape={table.shape}")
    print(table.isna().sum().to_string())

print("\nprimary key uniqueness")
print(f"  users.user_id   : {users['user_id'].is_unique}")
print(f"  movies.movie_id : {movies['movie_id'].is_unique}")

bad_user = ~ratings["user_id"].isin(users["user_id"])
bad_movie = ~ratings["movie_id"].isin(movies["movie_id"])

n_bad_user = bad_user.sum()
n_bad_movie = bad_movie.sum()
d_bad_user = ratings.loc[bad_user, "user_id"].nunique()
d_bad_movie = ratings.loc[bad_movie, "movie_id"].nunique()

print("\norphan foreign keys in ratings")
print(f"  rows, unknown user_id     : {n_bad_user:,}")
print(f"  rows, unknown movie_id    : {n_bad_movie:,}")
print(f"  distinct unknown user_id  : {d_bad_user}")
print(f"  distinct unknown movie_id : {d_bad_movie}")

STEP 2  missing values, keys and orphan rows

ratings: shape=(1000209, 4)
user_id      0
movie_id     0
rating       0
timestamp    0

users: shape=(6040, 5)
user_id       0
gender        0
age           0
occupation    0
zip           0

movies: shape=(3883, 3)
movie_id    0
title       0
genres      0

primary key uniqueness
  users.user_id   : True
  movies.movie_id : True

orphan foreign keys in ratings
  rows, unknown user_id     : 0
  rows, unknown movie_id    : 0
  distinct unknown user_id  : 0
  distinct unknown movie_id : 0


### Step 2b: Data Profile of the Three Raw Tables
Rows, columns, types, missing values, distinct values and ranges for each raw table (the profile table required by the brief).

In [3]:
print("STEP 2b  data profile of the raw tables")

def profile(table: pd.DataFrame, name: str) -> pd.DataFrame:
    """One row per column: dtype, missing, distinct values and numeric range."""
    rows = []
    for col in table.columns:
        s = table[col]
        numeric = pd.api.types.is_numeric_dtype(s)
        rows.append(
            {
                "column": col,
                "dtype": str(s.dtype),
                "missing": int(s.isna().sum()),
                "unique": int(s.nunique()),
                "min": s.min() if numeric else "-",
                "max": s.max() if numeric else "-",
            }
        )
    print(f"\n{name}: {table.shape[0]:,} rows x {table.shape[1]} columns")
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    return out

for name, table in [("ratings", ratings), ("users", users), ("movies", movies)]:
    profile(table, name)

STEP 2b  data profile of the raw tables

ratings: 1,000,209 rows x 4 columns
   column dtype  missing  unique       min        max
  user_id int64        0    6040         1       6040
 movie_id int64        0    3706         1       3952
   rating int64        0       5         1          5
timestamp int64        0  458455 956703932 1046454590

users: 6,040 rows x 5 columns
    column  dtype  missing  unique min  max
   user_id  int64        0    6040   1 6040
    gender object        0       2   -    -
       age  int64        0       7   1   56
occupation  int64        0      21   0   20
       zip object        0    3439   -    -

movies: 3,883 rows x 3 columns
  column  dtype  missing  unique min  max
movie_id  int64        0    3883   1 3952
   title object        0    3883   -    -
  genres object        0     301   -    -


### Step 2c: Duplicate, Validity and Coverage Checks
Duplicate rating pairs, valid value ranges, rating period, and films that no one rated.

In [4]:
print("STEP 2c  duplicate, validity and coverage checks")

n_dup_pairs = ratings.duplicated(subset=["user_id", "movie_id"]).sum()
n_dup_rows = ratings.duplicated().sum()
print("duplicate rows in ratings")
print(f"  repeated (user_id, movie_id) pairs : {n_dup_pairs:,}")
print(f"  fully identical rows               : {n_dup_rows:,}")

print("\nvalid value ranges")
print(f"  rating values : {[int(x) for x in sorted(ratings['rating'].unique())]}")
print(f"  gender codes  : {sorted(users['gender'].unique())}")
stamps = pd.to_datetime(ratings["timestamp"], unit="s", utc=True)
print(f"  first rating  : {stamps.min()}")
print(f"  last rating   : {stamps.max()}")

rated_ids = set(ratings["movie_id"])
n_unrated = (~movies["movie_id"].isin(rated_ids)).sum()
print("\nfilm coverage")
print(f"  films in movies.dat      : {len(movies):,}")
print(f"  films with >= 1 rating   : {len(rated_ids):,}")
print(f"  films never rated        : {n_unrated:,}")
print(f"  highest movie_id         : {movies['movie_id'].max():,}")
print(f"  unused ids up to highest : {movies['movie_id'].max() - len(movies):,}")
print(f"  fewest ratings by one user : {ratings.groupby('user_id').size().min()}")

STEP 2c  duplicate, validity and coverage checks
duplicate rows in ratings
  repeated (user_id, movie_id) pairs : 0
  fully identical rows               : 0

valid value ranges
  rating values : [1, 2, 3, 4, 5]
  gender codes  : ['F', 'M']
  first rating  : 2000-04-25 23:05:32+00:00
  last rating   : 2003-02-28 17:49:50+00:00

film coverage
  films in movies.dat      : 3,883
  films with >= 1 rating   : 3,706
  films never rated        : 177
  highest movie_id         : 3,952
  unused ids up to highest : 69
  fewest ratings by one user : 20


### Step 2d: Additional Checks (Data Analyst A)
Three checks taken from the independent notebook of Data Analyst A (Que): row counts against the README, ratings dated before the film's release year, and ratings entered in the same second by the same user. Nothing is removed here; the results are recorded in the cleaning log (C17). Two notebooks built separately give the same row counts and checks.

In [5]:
print("STEP 2d (1)  row counts against the README")

readme = re.sub(r"\s+", " ", zf.read("ml-1m/README").decode("latin-1"))
ref_ratings = int(re.search(r"([\d,]+) anonymous ratings", readme).group(1).replace(",", ""))
ref_users = int(re.search(r"made by ([\d,]+) MovieLens users", readme).group(1).replace(",", ""))

print(f"  ratings : README {ref_ratings:,}, file {len(ratings):,}, match {ref_ratings == len(ratings)}")
print(f"  users   : README {ref_users:,}, file {len(users):,}, match {ref_users == len(users)}")

STEP 2d (1)  row counts against the README
  ratings : README 1,000,209, file 1,000,209, match True
  users   : README 6,040, file 6,040, match True


In [6]:
print("STEP 2d (2)  ratings dated before the film's release year")

release_year = (
    movies.set_index("movie_id")["title"]
    .str.extract(r"\((\d{4})\)\s*$", expand=False)
    .astype(int)
)
rated_year = pd.to_datetime(ratings["timestamp"], unit="s", utc=True).dt.year
before = rated_year < ratings["movie_id"].map(release_year)

print(f"  ratings checked           : {len(ratings):,}")
print(f"  rated before release year : {before.sum():,}")

STEP 2d (2)  ratings dated before the film's release year
  ratings checked           : 1,000,209
  rated before release year : 0


In [7]:
print("STEP 2d (3)  ratings entered in the same second by the same user")

per_second = ratings.groupby(["user_id", "timestamp"]).size()
in_burst = int(per_second[per_second > 1].sum())

print(f"  ratings checked                         : {len(ratings):,}")
print(f"  ratings sharing a second with another   : {in_burst:,} ({in_burst / len(ratings):.1%})")
print(f"  largest burst, one user in one second   : {per_second.max()} ratings")

STEP 2d (3)  ratings entered in the same second by the same user
  ratings checked                         : 1,000,209
  ratings sharing a second with another   : 774,028 (77.4%)
  largest burst, one user in one second   : 30 ratings


### Step 3: Merge Into One Working Table
Merge the `ratings`, `users`, and `movies` DataFrames using left joins.

In [8]:
print("STEP 3  merge into one working table")

df = ratings.merge(users, on="user_id", how="left", validate="m:1")
df = df.merge(movies, on="movie_id", how="left", validate="m:1")

assert len(df) == len(ratings), "left merge changed the row count"

print(f"shape     : {df.shape}")
print(f"columns   : {df.columns.tolist()}")
print(f"total NaN : {df.isna().sum().sum()}")

STEP 3  merge into one working table
shape     : (1000209, 10)
columns   : ['user_id', 'movie_id', 'rating', 'timestamp', 'gender', 'age', 'occupation', 'zip', 'title', 'genres']
total NaN : 0


### Step 3b: Drop Personal Columns
`zip` and `occupation` are dropped from the working table: zip code combined with gender and age band could narrow down an individual, and neither column is used in the analysis (as committed in D1).

In [9]:
print("STEP 3b  drop personal columns")

before = df.shape[1]
df = df.drop(columns=["zip", "occupation"])
print(f"columns before : {before}")
print(f"columns after  : {df.shape[1]}")
print(f"columns        : {df.columns.tolist()}")
assert len(df) == len(ratings), "dropping columns changed the row count"

STEP 3b  drop personal columns
columns before : 10
columns after  : 8
columns        : ['user_id', 'movie_id', 'rating', 'timestamp', 'gender', 'age', 'title', 'genres']


### Step 4: Derived Columns (Year and Age Group)
Extract the release year from movie titles and map numeric age codes to readable categories.

In [10]:
print("STEP 4  derived columns, year and age_group")

df["year"] = (
    df["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).astype("Int64")
)

print(f"rows with NaN year : {df['year'].isna().sum()}")
print(f"year.min()         : {df['year'].min()}")
print(f"year.max()         : {df['year'].max()}")

codes = df["age"]
df["age_group"] = codes.map(AGE_LABEL)

assert df["age_group"].isna().sum() == 0, "unmapped age code found"

print(f"\nrows with NaN age_group : {df['age_group'].isna().sum()}")

age_order = [AGE_LABEL[code] for code in sorted(AGE_LABEL)]
print(df["age_group"].value_counts().reindex(age_order).to_string())

STEP 4  derived columns, year and age_group
rows with NaN year : 0
year.min()         : 1919
year.max()         : 2000

rows with NaN age_group : 0
age_group
Under 18     27211
18-24       183536
25-34       395556
35-44       199003
45-49        83633
50-55        72490
56+          38780


In [11]:
print("STEP 4b  release-year coverage")

years_rated = set(df["year"].dropna().astype(int))
years_all = set(
    movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).dropna().astype(int)
)
span = range(min(years_all), max(years_all) + 1)
print(f"distinct years, rated films in df : {len(years_rated)}")
print(f"distinct years, all of movies.dat : {len(years_all)}")
print(f"years in the span with no film    : {[y for y in span if y not in years_all]}")

STEP 4b  release-year coverage
distinct years, rated films in df : 81
distinct years, all of movies.dat : 81
years in the span with no film    : [1924]


### Step 5: Genre Breakdown
Explode the concatenated genres so each row represents a single rating-genre pair.

In [12]:
print("STEP 5  genre breakdown")

genres = df.copy()
genres["genre_list"] = genres["genres"].str.split("|")
genres["n_genres"] = genres["genre_list"].str.len()

print("n_genres per rating row")
print(genres["n_genres"].describe().to_string())

exploded = genres.explode("genre_list").rename(columns={"genre_list": "genre"})

print(f"\nrows after explode : {len(exploded):,}")
print(f"ratio to ratings   : {len(exploded) / len(ratings):.4f}")
print(f"distinct genres    : {exploded['genre'].nunique()}")
print(exploded["genre"].value_counts().to_string())

STEP 5  genre breakdown
n_genres per rating row
count    1.000209e+06
mean     2.101376e+00
std      9.768456e-01
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.000000e+00

rows after explode : 2,101,815
ratio to ratings   : 2.1014
distinct genres    : 18
genre
Comedy         356580
Drama          354529
Action         257457
Thriller       189680
Sci-Fi         157294
Romance        147523
Adventure      133953
Crime           79541
Horror          76386
Children's      72186
War             68527
Animation       43293
Musical         41533
Mystery         40178
Fantasy         36301
Western         20683
Film-Noir       18261
Documentary      7910


In [13]:
print("STEP 5b  labels per film and genre combinations")

per_film_all = movies["genres"].str.count(r"\|") + 1
per_film_rated = per_film_all[movies["movie_id"].isin(rated_ids)]
print(f"distinct genre combinations         : {movies['genres'].nunique()}")
print(f"mean labels per rating row (df)     : {genres['n_genres'].mean():.4f}")
print(f"mean labels per film, all movies    : {per_film_all.mean():.4f}")
print(f"mean labels per film, rated movies  : {per_film_rated.mean():.4f}")

STEP 5b  labels per film and genre combinations
distinct genre combinations         : 301
mean labels per rating row (df)     : 2.1014
mean labels per film, all movies    : 1.6503
mean labels per film, rated movies  : 1.6708


### Step 6: Cell Sparsity
Count distinct users and ratings in each of the 252 cells (segment × genre) to see which cells have thin data support.

In [14]:
print("STEP 6  cell sparsity")

cells = (
    exploded.groupby(["gender", "age_group", "genre"], observed=True)
    .agg(n_ratings=("rating", "size"), n_users=("user_id", "nunique"))
    .reset_index()
    .sort_values("n_users")
)

cells["age_group"] = pd.Categorical(
    cells["age_group"], age_order, ordered=True
)

print(f"cells present    : {len(cells)}")
print(f"full grid 2x7x18 : {2 * 7 * 18}")
print("\nn_users per cell")
print(cells["n_users"].describe().round(1).to_string())

for limit in (30, 50, 100):
    thin = cells[cells["n_users"] < limit]
    share = len(thin) / len(cells)
    print(
        f"\nunder {limit} users: {len(thin)} of {len(cells)} cells ({share:.1%})"
    )
    rows = zip(
        thin["gender"],
        thin["age_group"],
        thin["genre"],
        thin["n_users"],
        thin["n_ratings"],
    )
    for gender, age, genre, n_users, n_ratings in rows:
        label = f"{gender} {age}"
        print(
            f"  {label:<12} {genre:<12} {n_users:>3} users, {n_ratings:>5} ratings"
        )

STEP 6  cell sparsity
cells present    : 252
full grid 2x7x18 : 252

n_users per cell
count     252.0
mean      372.6
std       354.7
min        11.0
25%       133.5
50%       270.5
75%       460.5
max      1538.0

under 30 users: 3 of 252 cells (1.2%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings

under 50 users: 5 of 252 cells (2.0%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings
  F Under 18   Western       35 users,    91 ratings
  M Under 18   Documentary   45 users,   105 ratings

under 100 users: 41 of 252 cells (16.3%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings
  F Under 18   Western       35 users,    91 ratings
  M Under 18   Documentary   4

### Step 7: Freeze the Reliability Threshold into a Tier Column
A cell is **reliable** if it holds at least 100 distinct users, and **indicative** otherwise. The threshold was fixed in D1 (Section 5, Feasibility note) before any satisfaction rate was computed. Cell size is counted in distinct users, not ratings, because one person's ratings are not independent of each other. Indicative cells stay in the cell table and are reported as indicative only; they are never used to rank a segment's genres. D1 allowed such cells to be "pooled or reported as indicative only"; this notebook takes the second option and does not pool.

In [15]:
print("STEP 7  freeze the reliability threshold into a tier column")

MIN_USERS = 100  # fixed in D1, Section 5, before any satisfaction rate was computed

cells["tier"] = np.where(cells["n_users"] >= MIN_USERS, "reliable", "indicative")

tier_counts = cells["tier"].value_counts().reindex(["reliable", "indicative"])
assert tier_counts.sum() == len(cells) == 2 * 7 * 18, "tiers miss some cells"

print(f"threshold : {MIN_USERS} distinct users per cell")
print(tier_counts.to_string())
print(f"reliable share : {tier_counts['reliable'] / len(cells):.1%}")

print("\nreliable cells per segment (out of 18 genres)")
reliable_per_segment = (
    cells.groupby(["gender", "age_group"], observed=False)["tier"]
    .apply(lambda s: int((s == "reliable").sum()))
    .unstack("age_group")
)
print(reliable_per_segment.to_string())

STEP 7  freeze the reliability threshold into a tier column
threshold : 100 distinct users per cell
tier
reliable      211
indicative     41
reliable share : 83.7%

reliable cells per segment (out of 18 genres)
age_group  Under 18  18-24  25-34  35-44  45-49  50-55  56+
gender                                                     
F                 0     17     18     18     17     15    4
M                15     18     18     18     18     18   17


### Step 7b: Filter the Cells Used for Ranking
Data manipulation operation: filtering. Only reliable cells are kept in the table used to rank genres within a segment; indicative cells are set aside and reported separately.

In [16]:
print("STEP 7b  filter the cells used for ranking")

ranked = cells[cells["tier"] == "reliable"]
indicative = cells[cells["tier"] == "indicative"]
assert len(ranked) + len(indicative) == len(cells), "filter lost cells"

print(f"cells before filter            : {len(cells)}")
print(f"cells kept for ranking         : {len(ranked)}")
print(f"cells set aside as indicative  : {len(indicative)}")

STEP 7b  filter the cells used for ranking
cells before filter            : 252
cells kept for ranking         : 211
cells set aside as indicative  : 41


### Step 8: Satisfied Flag and Satisfaction Rate per Cell
Derived column `satisfied` (rating ≥ 4) and the satisfaction rate of each cell, as defined in D1 Section 2: the share of a cell's **ratings** that are 4 or 5 stars. No confidence interval is reported here: ratings from one user are not independent, so an interval that treats each rating as independent would be too narrow. User-level intervals are part of the D3 analysis. Rows below are sorted by segment and genre, not by rate; no genre is ranked in D2.

In [17]:
print("STEP 8  satisfied flag and satisfaction rate per cell")

exploded["satisfied"] = exploded["rating"] >= 4

rates = (
    exploded.groupby(["gender", "age_group", "genre"], observed=True)
    .agg(
        n_ratings=("rating", "size"),
        n_users=("user_id", "nunique"),
        n_satisfied=("satisfied", "sum"),
        sat_rate=("satisfied", "mean"),
    )
    .reset_index()
)

rates = rates.merge(
    cells[["gender", "age_group", "genre", "tier"]],
    on=["gender", "age_group", "genre"],
    how="left",
    validate="1:1",
)

assert len(rates) == len(cells), "cell count changed"
assert rates["sat_rate"].between(0, 1).all(), "sat_rate outside [0, 1]"
assert (rates["n_satisfied"] <= rates["n_ratings"]).all(), "more satisfied than ratings"

print(f"cells                                     : {len(rates)}")
print(f"share of ratings >= 4, all ratings, no explode : {(df['rating'] >= 4).mean():.4f}")
print("\nfirst rows of the cell table (sorted by segment and genre, not by rate)")
print(rates.sort_values(["gender", "age_group", "genre"]).head(5).to_string(index=False))

STEP 8  satisfied flag and satisfaction rate per cell
cells                                     : 252
share of ratings >= 4, all ratings, no explode : 0.5752

first rows of the cell table (sorted by segment and genre, not by rate)
gender age_group      genre  n_ratings  n_users  n_satisfied  sat_rate     tier
     F     18-24     Action       8593      296         4126  0.480158 reliable
     F     18-24  Adventure       5430      291         2611  0.480847 reliable
     F     18-24  Animation       3109      259         1806  0.580894 reliable
     F     18-24 Children's       5234      276         2575  0.491976 reliable
     F     18-24     Comedy      19136      298         9878  0.516200 reliable


### Step 9: Data Profile of the Working Table
Rows, columns, types, missing values, distinct values and ranges for the merged table `df` (after dropping personal columns and adding `year` and `age_group`).

In [18]:
print("STEP 9  data profile of the working table")

_ = profile(df, "df (working table)")

STEP 9  data profile of the working table

df (working table): 1,000,209 rows x 10 columns
   column  dtype  missing  unique       min        max
  user_id  int64        0    6040         1       6040
 movie_id  int64        0    3706         1       3952
   rating  int64        0       5         1          5
timestamp  int64        0  458455 956703932 1046454590
   gender object        0       2         -          -
      age  int64        0       7         1         56
    title object        0    3706         -          -
   genres object        0     301         -          -
     year  Int64        0      81      1919       2000
age_group object        0       7         -          -


## D3 analysis

### Step B1: Split Users into a Discovery Half and a Validation Half
Within each of the 14 segments, users are shuffled with seed 42 and split 50/50. The split is made on users, not on ratings, so all ratings of one person stay in the same half (ratings from one user are not independent). When a segment has an odd number of users, the extra user goes to the validation half. Genres are chosen on the discovery half (Steps B4 to B7) and tested on the validation half (Step B8), so the noise that helped a genre get selected is not counted again in the test. Only counts are printed here; no satisfaction rate is computed per half in this Step. Tiers stay as frozen in Step 7 on the full data.

In [19]:
print("STEP B1  split users into a discovery half and a validation half")

rng = np.random.default_rng(SEED)

seg_users = users[["user_id", "gender", "age"]].copy()
seg_users["age_group"] = seg_users["age"].map(AGE_LABEL)

half_of: dict[int, str] = {}
for gender in ["F", "M"]:              # fixed order, so the random draws are reproducible
    for age in age_order:
        in_seg = (seg_users["gender"] == gender) & (seg_users["age_group"] == age)
        ids = np.sort(seg_users.loc[in_seg, "user_id"].to_numpy())
        shuffled = rng.permutation(ids)
        n_discovery = len(ids) // 2    # odd count: the extra user goes to validation
        for uid in shuffled[:n_discovery]:
            half_of[int(uid)] = "discovery"
        for uid in shuffled[n_discovery:]:
            half_of[int(uid)] = "validation"

seg_users["half"] = seg_users["user_id"].map(half_of)
df["half"] = df["user_id"].map(half_of)
exploded["half"] = exploded["user_id"].map(half_of)

assert len(half_of) == len(users) == seg_users["user_id"].nunique(), "a user is missing or repeated"
assert seg_users["half"].notna().all(), "a user has no half"
assert df["half"].notna().all() and exploded["half"].notna().all(), "a rating has no half"
assert (df.groupby("user_id")["half"].nunique() == 1).all(), "a user appears in both halves"

users_per_half = (
    seg_users.groupby(["gender", "age_group", "half"]).size()
    .unstack("half")
    .reindex(pd.MultiIndex.from_product([["F", "M"], age_order], names=["gender", "age_group"]))
)
users_per_half["difference"] = users_per_half["validation"] - users_per_half["discovery"]
assert users_per_half["difference"].isin([0, 1]).all(), "halves differ by more than one user"

print("users per segment and half (difference = validation - discovery)")
print(users_per_half.to_string())
print(f"\nusers, discovery  : {(seg_users['half'] == 'discovery').sum():,}")
print(f"users, validation : {(seg_users['half'] == 'validation').sum():,}")
print(f"users, total      : {len(seg_users):,}")

ratings_per_half = df["half"].value_counts().reindex(["discovery", "validation"])
print("\nratings per half (working table df)")
for half, n in ratings_per_half.items():
    print(f"  {half:<10} : {n:,}")
print(f"  total      : {ratings_per_half.sum():,}")

STEP B1  split users into a discovery half and a validation half
users per segment and half (difference = validation - discovery)
half              discovery  validation  difference
gender age_group                                   
F      Under 18          39          39           0
       18-24            149         149           0
       25-34            279         279           0
       35-44            169         169           0
       45-49             94          95           1
       50-55             73          73           0
       56+               51          51           0
M      Under 18          72          72           0
       18-24            402         403           1
       25-34            769         769           0
       35-44            427         428           1
       45-49            180         181           1
       50-55            175         175           0
       56+              139         139           0

users, discovery  : 3,018
users, vali

### Step B2: Count n(u, g) and k(u, g) per User and Genre
For every user and every genre: n = number of ratings the user gave to films of that genre, k = number of those ratings with 4 or 5 stars. A user who rated no film of a genre gets n = k = 0. The table has one row per user and genre (6,040 x 18 rows) and keeps the half of each user.

In [20]:
print("STEP B2  user x genre counts n(u, g) and k(u, g)")

GENRES = sorted(exploded["genre"].unique())

per_user = exploded.groupby(["user_id", "genre"]).agg(
    n=("rating", "size"), k=("satisfied", "sum")
)
full_index = pd.MultiIndex.from_product(
    [seg_users["user_id"].sort_values(), GENRES], names=["user_id", "genre"]
)
user_genre = per_user.reindex(full_index, fill_value=0).reset_index()
user_genre["k"] = user_genre["k"].astype(int)
user_genre = user_genre.merge(
    seg_users[["user_id", "gender", "age_group", "half"]],
    on="user_id", how="left", validate="m:1",
)

assert len(user_genre) == len(users) * len(GENRES)
assert user_genre["n"].sum() == len(exploded)
assert user_genre["k"].sum() == exploded["satisfied"].sum()
assert (user_genre["k"] <= user_genre["n"]).all()

print(f"rows : {len(user_genre):,} ({len(users):,} users x {len(GENRES)} genres)")
print(f"sum n: {user_genre['n'].sum():,}   sum k: {user_genre['k'].sum():,}")
print(f"rows with n = 0 : {(user_genre['n'] == 0).sum():,}")
print(user_genre.head(3).to_string(index=False))

STEP B2  user x genre counts n(u, g) and k(u, g)
rows : 108,720 (6,040 users x 18 genres)
sum n: 2,101,815   sum k: 1,206,999
rows with n = 0 : 14,833
 user_id     genre  n  k gender age_group      half
       1    Action  5  4      F  Under 18 discovery
       1 Adventure  5  4      F  Under 18 discovery
       1 Animation 18 14      F  Under 18 discovery


### Step B3: Satisfaction Rate per Cell and Half
r(s, g) = sum of k / sum of n over the users of segment s, computed separately on the discovery half and the validation half. The table also carries the number of users, the tier from Step 7 (all data), and the reach of each cell. Column names follow D3_columns_HoangQuan_v2, Table 1.

In [21]:
print("STEP B3  satisfaction rate per cell and half")

rate_table = (
    user_genre.groupby(["gender", "age_group", "genre", "half"])
    .agg(
        n_ratings=("n", "sum"),
        n_satisfied=("k", "sum"),
        n_users=("n", lambda s: int((s > 0).sum())),
    )
    .reset_index()
)
rate_table["sat_rate"] = rate_table["n_satisfied"] / rate_table["n_ratings"]

tier_all = (
    cells[["gender", "age_group", "genre", "n_users", "tier"]]
    .rename(columns={"n_users": "n_users_all"})
    .astype({"age_group": str})
)
rate_table = rate_table.merge(
    tier_all, on=["gender", "age_group", "genre"], how="left", validate="m:1"
)

seg_size = (
    seg_users.groupby(["gender", "age_group", "half"]).size()
    .rename("segment_users").reset_index()
)
rate_table = rate_table.merge(
    seg_size, on=["gender", "age_group", "half"], how="left", validate="m:1"
)
rate_table["reach"] = rate_table["n_users"] / rate_table["segment_users"]

rate_table = rate_table[[
    "gender", "age_group", "genre", "half", "n_ratings", "n_users",
    "n_satisfied", "sat_rate", "n_users_all", "tier", "segment_users", "reach",
]]

assert len(rate_table) == 2 * 7 * 18 * 2
assert rate_table["sat_rate"].between(0, 1).all()
assert rate_table["tier"].notna().all() and rate_table["segment_users"].notna().all()

print(f"rows: {len(rate_table)}")
print(rate_table.dtypes.to_string())
check = (rate_table["gender"] == "F") & (rate_table["age_group"] == "18-24") & (rate_table["genre"] == "Action")
print(rate_table[check].to_string(index=False))

STEP B3  satisfaction rate per cell and half
rows: 504
gender            object
age_group         object
genre             object
half              object
n_ratings          int64
n_users            int64
n_satisfied        int64
sat_rate         float64
n_users_all        int64
tier              object
segment_users      int64
reach            float64
gender age_group  genre       half  n_ratings  n_users  n_satisfied  sat_rate  n_users_all     tier  segment_users    reach
     F     18-24 Action  discovery       4196      149         2128  0.507150          296 reliable            149 1.000000
     F     18-24 Action validation       4397      147         1998  0.454401          296 reliable            149 0.986577


### Step B4: Lower Confidence Bound (LCB) of Each Cell, Discovery Half
For each of the 14 segments, the users of the discovery half are resampled with replacement 1,000 times (whole users, not single ratings). In every round the rate of each genre is recomputed as sum of k / sum of n over the resampled users. LCB(s, g) is the 2.5% quantile of the 1,000 rates. The random generator is created at the start of this Step with seed 42.

In [22]:
print("STEP B4  LCB of each cell, discovery half")

N_BOOT = 1000
rng = np.random.default_rng(SEED)

disc = user_genre[user_genre["half"] == "discovery"]
parts = []
for gender in ["F", "M"]:            # fixed order, so the random draws are reproducible
    for age in age_order:
        seg = disc[(disc["gender"] == gender) & (disc["age_group"] == age)]
        n_mat = seg.pivot(index="user_id", columns="genre", values="n")[GENRES].to_numpy()
        k_mat = seg.pivot(index="user_id", columns="genre", values="k")[GENRES].to_numpy()
        n_users_seg = len(n_mat)
        pick = rng.integers(0, n_users_seg, size=(N_BOOT, n_users_seg))
        n_sum = n_mat[pick].sum(axis=1)
        k_sum = k_mat[pick].sum(axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            boot_rates = k_sum / n_sum
        parts.append(pd.DataFrame({
            "gender": gender, "age_group": age, "genre": GENRES,
            "lcb": np.nanquantile(boot_rates, 0.025, axis=0),
        }))

lcb_table = pd.concat(parts, ignore_index=True)
disc_rates = rate_table[rate_table["half"] == "discovery"][
    ["gender", "age_group", "genre", "tier", "n_users", "sat_rate"]
]
lcb_table = disc_rates.merge(
    lcb_table, on=["gender", "age_group", "genre"], how="left", validate="1:1"
)

assert len(lcb_table) == 2 * 7 * 18
assert lcb_table["lcb"].notna().all()
assert (lcb_table["lcb"] <= lcb_table["sat_rate"]).all()

print(f"cells: {len(lcb_table)}; bootstrap rounds: {N_BOOT}")
f1824 = (lcb_table["gender"] == "F") & (lcb_table["age_group"] == "18-24")
print(lcb_table[f1824].sort_values("lcb", ascending=False).head(4).to_string(index=False))

STEP B4  LCB of each cell, discovery half
cells: 252; bootstrap rounds: 1000
gender age_group       genre       tier  n_users  sat_rate      lcb
     F     18-24 Documentary indicative       47  0.770492 0.679218
     F     18-24   Film-Noir   reliable       86  0.704981 0.624986
     F     18-24         War   reliable      142  0.664103 0.619972
     F     18-24     Musical   reliable      124  0.652962 0.600511


### Step B5: Rank Cells by LCB and Choose T(s)
Within each segment, only reliable cells are ranked by LCB (highest first). Ties are broken by the number of users in the cell (more first), then by genre name. T(s) is the set of the three reliable cells with the highest LCB. Indicative cells get no rank and are never in T(s). Column names follow D3_columns_HoangQuan_v2, Table 2.

In [23]:
print("STEP B5  rank cells by LCB and choose T(s)")

lcb_table["age_order"] = lcb_table["age_group"].map({a: i for i, a in enumerate(age_order)})
reliable = lcb_table[lcb_table["tier"] == "reliable"].sort_values(
    ["gender", "age_order", "lcb", "n_users", "genre"],
    ascending=[True, True, False, False, True],
)
reliable["rank_lcb"] = reliable.groupby(["gender", "age_group"]).cumcount() + 1

lcb_table = lcb_table.merge(
    reliable[["gender", "age_group", "genre", "rank_lcb"]],
    on=["gender", "age_group", "genre"], how="left", validate="1:1",
)
lcb_table["rank_lcb"] = lcb_table["rank_lcb"].astype("Int64")
lcb_table["in_top3"] = lcb_table["rank_lcb"].le(3).fillna(False).astype(bool)
lcb_table = lcb_table.sort_values(["gender", "age_order", "genre"]).reset_index(drop=True)
lcb_table = lcb_table[["gender", "age_group", "genre", "tier", "sat_rate", "lcb", "rank_lcb", "in_top3"]]

assert len(lcb_table) == 252
assert lcb_table.loc[lcb_table["tier"] == "indicative", "rank_lcb"].isna().all()
assert lcb_table[lcb_table["in_top3"]].groupby(["gender", "age_group"]).size().max() == 3

t_s = (
    lcb_table[lcb_table["in_top3"]]
    .sort_values(["gender", "age_group", "rank_lcb"])
    .groupby(["gender", "age_group"])["genre"].apply(lambda s: "; ".join(s))
)
print(f"segments with a T(s): {len(t_s)} of 14")
print(t_s.to_string())
print(lcb_table.dtypes.to_string())

STEP B5  rank cells by LCB and choose T(s)
segments with a T(s): 13 of 14
gender  age_group
F       18-24            Film-Noir; War; Musical
        25-34        Film-Noir; Documentary; War
        35-44            Film-Noir; War; Musical
        45-49                War; Musical; Drama
        50-55                War; Drama; Romance
        56+             Drama; Romance; Thriller
M       18-24        Film-Noir; War; Documentary
        25-34        Film-Noir; Documentary; War
        35-44        Film-Noir; Documentary; War
        45-49              Film-Noir; War; Crime
        50-55            Film-Noir; War; Mystery
        56+              Film-Noir; War; Musical
        Under 18               War; Drama; Crime
gender        object
age_group     object
genre         object
tier          object
sat_rate     float64
lcb          float64
rank_lcb       Int64
in_top3         bool


### Step B6: Baseline B (Weighted Score of the Common Campaign), Discovery Half
B is the set of three genres promoted to everyone. It uses the 12 reference segments (segments with at least 15 reliable cells) and the 15 candidate genres (the 18 genres without Documentary, Film-Noir and Western). Each segment has a weight w(s) = N(s) / sum of N over the 12 segments, where N(s) is the number of users of the segment in the full users table. The score of a genre is S(g) = sum of w(s) x r(s, g), with r(s, g) the discovery-half rate. LCB of S(g) is the 2.5% quantile of 1,000 bootstrap rounds (users resampled within each segment, weights kept fixed). B = the three genres with the highest LCB; ties are broken by the number of users, then by genre name. Column names follow D3_columns_HoangQuan_v2, Table 3.


In [24]:
print("STEP B6  baseline B (weighted, discovery half)")

EXCLUDED = ["Documentary", "Film-Noir", "Western"]
CANDIDATES = [g for g in GENRES if g not in EXCLUDED]

reliable_count = (
    cells[cells["tier"] == "reliable"].groupby(["gender", "age_group"], observed=True).size()
)
ref_segments = [
    (g, a) for g in ["F", "M"] for a in age_order if reliable_count.get((g, a), 0) >= 15
]

seg_n = seg_users.groupby(["gender", "age_group"]).size()
weights = {s: seg_n[s] / sum(seg_n[t] for t in ref_segments) for s in ref_segments}

tier_lookup = tier_all.set_index(["gender", "age_group", "genre"])["tier"]
assert all(
    tier_lookup[(g, a, c)] == "reliable" for (g, a) in ref_segments for c in CANDIDATES
)

rng = np.random.default_rng(SEED)
disc = user_genre[user_genre["half"] == "discovery"]
s_point = np.zeros(len(CANDIDATES))
s_boot = np.zeros((N_BOOT, len(CANDIDATES)))
n_users_total = np.zeros(len(CANDIDATES))
for (gender, age) in ref_segments:
    seg = disc[(disc["gender"] == gender) & (disc["age_group"] == age)]
    n_mat = seg.pivot(index="user_id", columns="genre", values="n")[CANDIDATES].to_numpy()
    k_mat = seg.pivot(index="user_id", columns="genre", values="k")[CANDIDATES].to_numpy()
    w = weights[(gender, age)]
    s_point += w * k_mat.sum(axis=0) / n_mat.sum(axis=0)
    pick = rng.integers(0, len(n_mat), size=(N_BOOT, len(n_mat)))
    s_boot += w * k_mat[pick].sum(axis=1) / n_mat[pick].sum(axis=1)
    n_users_total += (n_mat > 0).sum(axis=0)

baseline_table = pd.DataFrame({
    "genre": CANDIDATES,
    "s_g": s_point,
    "lcb": np.quantile(s_boot, 0.025, axis=0),
    "n_users_total": n_users_total,
})
baseline_table = baseline_table.sort_values(
    ["lcb", "n_users_total", "genre"], ascending=[False, False, True]
).reset_index(drop=True)
baseline_table["rank_lcb"] = baseline_table.index + 1
baseline_table["in_baseline"] = baseline_table["rank_lcb"] <= 3
baseline_table = baseline_table[["genre", "s_g", "lcb", "rank_lcb", "in_baseline"]]

assert len(ref_segments) == 12 and len(baseline_table) == 15
assert abs(sum(weights.values()) - 1) < 1e-12
assert baseline_table["in_baseline"].sum() == 3
B = baseline_table.loc[baseline_table["in_baseline"], "genre"].tolist()
print(f"reference segments: {len(ref_segments)}; candidate genres: {len(CANDIDATES)}")
print(baseline_table.to_string(index=False))
print("B =", "; ".join(B))

STEP B6  baseline B (weighted, discovery half)
reference segments: 12; candidate genres: 15
     genre      s_g      lcb  rank_lcb  in_baseline
       War 0.690715 0.680999         1         True
     Drama 0.640402 0.629756         2         True
     Crime 0.619416 0.608235         3         True
 Animation 0.619236 0.606261         4        False
   Musical 0.600899 0.586528         5        False
   Mystery 0.593810 0.579815         6        False
   Romance 0.574833 0.563578         7        False
  Thriller 0.564373 0.553793         8        False
    Comedy 0.553472 0.543199         9        False
    Action 0.536783 0.526770        10        False
 Adventure 0.531281 0.520986        11        False
    Sci-Fi 0.524124 0.514298        12        False
Children's 0.522101 0.508674        13        False
   Fantasy 0.515268 0.503547        14        False
    Horror 0.438634 0.425255        15        False
B = War; Drama; Crime


### Step B7: Classify the 14 Segments and Count m
Before any test, each segment gets one of three labels, using only the tier and the discovery half (section 3.8 of the shared card): insufficient_evidence (the segment has no reliable cell, or a genre of B is indicative in the segment), same_as_baseline (T(s) equals B exactly, so there is nothing to test), or tested (all others). m is the number of segments to be tested. Step B9 turns "tested" into passed or not_passed after the Holm correction.

In [25]:
print("STEP B7  classify segments and count m")

rows = []
for gender in ["F", "M"]:
    for age in age_order:
        seg_cells = lcb_table[(lcb_table["gender"] == gender) & (lcb_table["age_group"] == age)]
        t_list = seg_cells[seg_cells["in_top3"]].sort_values("rank_lcb")["genre"].tolist()
        indicative_b = [g for g in B if tier_lookup[(gender, age, g)] == "indicative"]
        if len(t_list) == 0:
            category, reason = "insufficient_evidence", "no reliable cell in the segment"
        elif indicative_b:
            category = "insufficient_evidence"
            reason = "genre of B is indicative in this segment: " + "; ".join(indicative_b)
        elif set(t_list) == set(B):
            category, reason = "same_as_baseline", "T(s) equals B, no test"
        else:
            category, reason = "tested", ""
        rows.append({
            "segment": f"{gender} {age}", "gender": gender, "age_group": age,
            "category": category, "reason": reason, "t_s": "; ".join(t_list),
        })

segment_status = pd.DataFrame(rows)
m = int((segment_status["category"] == "tested").sum())

assert len(segment_status) == 14
assert segment_status["category"].isin(["insufficient_evidence", "same_as_baseline", "tested"]).all()

print(segment_status[["segment", "category", "reason", "t_s"]].to_string(index=False))
print(f"\nm (segments to test) : {m}")
print(segment_status["category"].value_counts().to_string())

STEP B7  classify segments and count m
   segment              category                                               reason                         t_s
F Under 18 insufficient_evidence                      no reliable cell in the segment                            
   F 18-24                tested                                                          Film-Noir; War; Musical
   F 25-34                tested                                                      Film-Noir; Documentary; War
   F 35-44                tested                                                          Film-Noir; War; Musical
   F 45-49                tested                                                              War; Musical; Drama
   F 50-55                tested                                                              War; Drama; Romance
     F 56+ insufficient_evidence genre of B is indicative in this segment: War; Crime    Drama; Romance; Thriller
M Under 18      same_as_baseline                 

### Step B8: D(s), Confidence Interval and Raw p-value, Validation Half
For each tested segment, on the validation half: mean r over the three genres of T(s) minus mean r over the three genres of B, where r is the segment's satisfaction rate of the genre (sum of k / sum of n). D(s) is the difference. The users of the segment are resampled with replacement 10,000 times; the 95% interval is [Q0.025, Q0.975] of the resampled D, without correction for several tests. The two-sided p-value is p = 2 x min(share of D* <= 0, share of D* >= 0), capped at 1 (null hypothesis: D(s) = 0). The random generator is created at the start of this Step with seed 42.

In [26]:
print("STEP B8  D(s), confidence interval and raw p-value, validation half")

N_BOOT_TEST = 10_000
CHUNK = 1_000
rng = np.random.default_rng(SEED)

valid = user_genre[user_genre["half"] == "validation"]
test_rows = []
for _, seg_row in segment_status[segment_status["category"] == "tested"].iterrows():
    gender, age = seg_row["gender"], seg_row["age_group"]
    t_genres = seg_row["t_s"].split("; ")
    seg = valid[(valid["gender"] == gender) & (valid["age_group"] == age)]
    n_mat = seg.pivot(index="user_id", columns="genre", values="n")[GENRES].to_numpy()
    k_mat = seg.pivot(index="user_id", columns="genre", values="k")[GENRES].to_numpy()
    t_idx = [GENRES.index(g) for g in t_genres]
    b_idx = [GENRES.index(g) for g in B]

    def d_value(n_sum, k_sum):
        rate = k_sum / n_sum
        return rate[..., t_idx].mean(axis=-1) - rate[..., b_idx].mean(axis=-1)

    rate_all = k_mat.sum(axis=0) / n_mat.sum(axis=0)
    mean_r_ts = rate_all[t_idx].mean()
    mean_r_b = rate_all[b_idx].mean()
    d = mean_r_ts - mean_r_b

    boot_d = []
    for _ in range(N_BOOT_TEST // CHUNK):
        pick = rng.integers(0, len(n_mat), size=(CHUNK, len(n_mat)))
        boot_d.append(d_value(n_mat[pick].sum(axis=1), k_mat[pick].sum(axis=1)))
    boot_d = np.concatenate(boot_d)

    p_raw = min(1.0, 2 * min((boot_d <= 0).mean(), (boot_d >= 0).mean()))
    test_rows.append({
        "segment": seg_row["segment"], "mean_r_ts": mean_r_ts, "mean_r_b": mean_r_b, "d": d,
        "ci_low": np.quantile(boot_d, 0.025), "ci_high": np.quantile(boot_d, 0.975), "p_raw": p_raw,
    })

test_table = pd.DataFrame(test_rows)
assert len(test_table) == m
assert test_table["p_raw"].between(0, 1).all()
assert (test_table["ci_low"] <= test_table["ci_high"]).all()
print(f"segments tested: {len(test_table)}; bootstrap rounds per segment: {N_BOOT_TEST:,}")
print(test_table.round(4).to_string(index=False))

STEP B8  D(s), confidence interval and raw p-value, validation half
segments tested: 11; bootstrap rounds per segment: 10,000
segment  mean_r_ts  mean_r_b       d  ci_low  ci_high  p_raw
F 18-24     0.6000    0.5662  0.0338  0.0045   0.0619 0.0252
F 25-34     0.7190    0.6414  0.0776  0.0427   0.1069 0.0000
F 35-44     0.6872    0.6491  0.0381  0.0141   0.0616 0.0024
F 45-49     0.6344    0.6387 -0.0044 -0.0390   0.0280 0.8616
F 50-55     0.6710    0.6950 -0.0239 -0.0468   0.0030 0.0818
M 18-24     0.7322    0.6620  0.0702  0.0461   0.0917 0.0000
M 25-34     0.7283    0.6439  0.0844  0.0708   0.0974 0.0000
M 35-44     0.7392    0.6630  0.0762  0.0572   0.0942 0.0000
M 45-49     0.7244    0.6826  0.0418  0.0302   0.0528 0.0000
M 50-55     0.7289    0.6729  0.0560  0.0380   0.0723 0.0000
  M 56+     0.7372    0.7166  0.0206 -0.0043   0.0436 0.1004


### Step B9: Holm Correction and Final Results Table
The raw p-values of the m tested segments are sorted from smallest to largest. The j-th p-value passes if it is at most 0.05 / (m - j + 1) and every earlier p-value passed (the procedure stops at the first failure). p_holm is the adjusted p-value. A segment is "passed" only if it passes Holm and D(s) > 0; otherwise a tested segment is "not_passed". The adjusted p-values and decisions are checked against statsmodels.stats.multitest.multipletests(method="holm"). The 14-row results table follows D3_columns_HoangQuan_v2, Table 4.

In [27]:
print("STEP B9  Holm correction and final results table")

ALPHA = 0.05
order = test_table.sort_values("p_raw", kind="stable").reset_index(drop=True)
holm_pass_flags, p_holm_values = [], []
running_max, still_passing = 0.0, True
for j, p in enumerate(order["p_raw"], start=1):
    threshold = ALPHA / (m - j + 1)
    still_passing = still_passing and (p <= threshold)
    running_max = max(running_max, min(1.0, (m - j + 1) * p))
    holm_pass_flags.append(still_passing)
    p_holm_values.append(running_max)
order["p_holm"] = p_holm_values
order["holm_pass"] = holm_pass_flags

from statsmodels.stats.multitest import multipletests
reject, p_adj, _, _ = multipletests(order["p_raw"], alpha=ALPHA, method="holm")
assert np.allclose(order["p_holm"], p_adj), "Holm p-values differ from statsmodels"
assert (order["holm_pass"].to_numpy() == reject).all(), "Holm decisions differ from statsmodels"

order["passed"] = order["holm_pass"] & (order["d"] > 0)
results = segment_status.merge(
    order[["segment", "mean_r_ts", "mean_r_b", "d", "ci_low", "ci_high", "p_raw", "p_holm", "passed"]],
    on="segment", how="left", validate="1:1",
)
is_tested = results["category"] == "tested"
results.loc[is_tested, "category"] = np.where(
    results.loc[is_tested, "passed"].astype(bool), "passed", "not_passed"
)
results["m"] = m
results["conclusion"] = np.where(results["category"] == "passed", "own_campaign", "common_campaign")
results = results[[
    "segment", "gender", "age_group", "category", "reason", "t_s",
    "mean_r_ts", "mean_r_b", "d", "ci_low", "ci_high",
    "p_raw", "p_holm", "m", "conclusion",
]]

assert len(results) == 14
assert results["category"].isin(["insufficient_evidence", "same_as_baseline", "passed", "not_passed"]).all()
assert m == results["category"].isin(["passed", "not_passed"]).sum()

print(results[["segment", "category", "d", "p_raw", "p_holm", "conclusion"]].round(4).to_string(index=False))
print(f"\nm = {m}")
print(results["category"].value_counts().to_string())
print(results.dtypes.to_string())

STEP B9  Holm correction and final results table
   segment              category       d  p_raw  p_holm      conclusion
F Under 18 insufficient_evidence     NaN    NaN     NaN common_campaign
   F 18-24            not_passed  0.0338 0.0252  0.1008 common_campaign
   F 25-34                passed  0.0776 0.0000  0.0000    own_campaign
   F 35-44                passed  0.0381 0.0024  0.0120    own_campaign
   F 45-49            not_passed -0.0044 0.8616  0.8616 common_campaign
   F 50-55            not_passed -0.0239 0.0818  0.2454 common_campaign
     F 56+ insufficient_evidence     NaN    NaN     NaN common_campaign
M Under 18      same_as_baseline     NaN    NaN     NaN common_campaign
   M 18-24                passed  0.0702 0.0000  0.0000    own_campaign
   M 25-34                passed  0.0844 0.0000  0.0000    own_campaign
   M 35-44                passed  0.0762 0.0000  0.0000    own_campaign
   M 45-49                passed  0.0418 0.0000  0.0000    own_campaign
   M 50-55     

## Data manipulation operations (index)
| Operation | Where | What it does |
|-----------|-------|--------------|
| Merging two tables | Step 3 | left joins ratings, users and movies with validate="m:1" and asserts the row count |
| Creating a derived column | Steps 4, 7, 8 | year and age_group (Step 4), tier (Step 7), satisfied flag (Step 8) |
| Filtering | Step 7b | keeps the reliable cells (≥ 100 distinct users) used for ranking, counts printed before and after |
| Grouping and aggregation | Steps 6, 8 | users, ratings and satisfaction rate per gender x age_group x genre cell |

Step 5 (explode genres into one row per rating-genre pair) prepares the data for Steps 6 and 8 and is not counted as a fifth operation.

### Things that do NOT change the row count
- Steps 5 and 6 never modify `df`; they build `exploded` and the cell table from it. Cell sizes are counted in distinct users, and a user who rated several genres is counted in several cells of the same segment, so the cell sizes inside one segment do not add up to that segment's number of users.

## Inputs for the other roles
Environment and column lists printed from this notebook, so teammates take them from here instead of typing them by hand.

In [28]:
print("HANDOFF  environment and column lists")

import platform
from datetime import date
from importlib.metadata import version

print(f"run date: {date.today().isoformat()}")
print(f"python  : {platform.python_version()}")
for pkg in ("numpy", "pandas"):
    print(f"{pkg:<8}: {version(pkg)}")
print(f"seed    : {SEED}")

print("\nworking table df, columns")
for col, dtype in df.dtypes.items():
    print(f"  {col:<12} {dtype}")

print("\nexploded table, columns")
for col, dtype in exploded.dtypes.items():
    print(f"  {col:<12} {dtype}")

print("\ncell table rates, columns")
for col, dtype in rates.dtypes.items():
    print(f"  {col:<12} {dtype}")

HANDOFF  environment and column lists
run date: 2026-11-18
python  : 3.13.18
numpy   : 2.4.6
pandas  : 2.2.3
seed    : 42

working table df, columns
  user_id      int64
  movie_id     int64
  rating       int64
  timestamp    int64
  gender       object
  age          int64
  title        object
  genres       object
  year         Int64
  age_group    object
  half         object

exploded table, columns
  user_id      int64
  movie_id     int64
  rating       int64
  timestamp    int64
  gender       object
  age          int64
  title        object
  genres       object
  year         Int64
  age_group    object
  genre        object
  n_genres     int64
  satisfied    bool
  half         object

cell table rates, columns
  gender       object
  age_group    object
  genre        object
  n_ratings    int64
  n_users      int64
  n_satisfied  int64
  sat_rate     float64
  tier         object


## Cleaning log

Every figure below is printed by the Step named in the "Step" column. Rows C1 to C17 come from the D2 run of 2026-10-20 and every rerun must print the same figures; C18 and C19 were added in D3. Row counts refer to the 1,000,209-row ratings table unless stated otherwise.

| ID | Step | Check | Finding | Decision | Reason | Before → After |
|----|------|-------|---------|----------|--------|----------------|
| C1 | 2 | Missing values in `ratings`, `users`, `movies` | 0 missing in every column of all three tables | Keep as is | Nothing to impute or drop | 1,000,209 / 6,040 / 3,883 rows → unchanged |
| C2 | 2 | Primary key uniqueness | `users.user_id` and `movies.movie_id` are unique | Keep as is | Joins are many-to-one, so no row can multiply | unchanged |
| C3 | 2 | Orphan foreign keys in `ratings` | 0 rows with an unknown `user_id`, 0 with an unknown `movie_id` | Keep as is | Every rating can be joined | unchanged |
| C4 | 2c | Duplicate ratings | 0 repeated (`user_id`, `movie_id`) pairs, 0 fully identical rows | Nothing removed | No duplicates exist | 1,000,209 → 1,000,209 |
| C5 | 2c | Valid value ranges | `rating` ∈ {1,2,3,4,5}; `gender` ∈ {F, M}; ratings dated 2000-04-25 to 2003-02-28 | Keep as is | All values are inside the documented domain | unchanged |
| C6 | 2c | Films nobody rated | 3,883 films in `movies.dat`, 3,706 with at least 1 rating, 177 never rated | Left joins start from `ratings`, so the 177 films do not appear in the working table | A film with no rating carries no preference signal | 3,883 → 3,706 films in `df` |
| C7 | 3 | Merge `ratings` + `users` + `movies` | Left joins with `validate="m:1"`; row count asserted equal to `ratings` | Merged into one working table | One row per rating with user and film attributes | 1,000,209 → 1,000,209 rows; 4 → 10 columns |
| C8 | 3b | Personal columns | `zip` (3,439 distinct) could single out a person when combined with gender and age band; `occupation` is not used in the analysis | Dropped `zip` and `occupation` | Privacy and relevance, as committed in D1 | 10 → 8 columns |
| C9 | 4 / 4b | Release year parsed from `title` | 0 unparsed titles; years 1919–2000; 81 distinct years in rated films and in `movies.dat`; the only year without a film inside the span is 1924 | Added `year` | Needed for any release-period cut | 8 → 9 columns |
| C10 | 4 | Age code to label | 0 unmapped codes; 7 groups: Under 18 (27,211), 18-24 (183,536), 25-34 (395,556), 35-44 (199,003), 45-49 (83,633), 50-55 (72,490), 56+ (38,780); counts sum to 1,000,209 | Added `age_group` | Readable and ordered labels for the age groups | 9 → 10 columns |
| C11 | 5 | Multi-genre films | A rating row carries 1–6 genre labels (mean 2.1014); 301 distinct genre strings; 18 distinct genres | Exploded to one row per rating-genre pair, in a separate table `exploded`; `df` is untouched | The unit of analysis is the genre | 1,000,209 → 2,101,815 rows (×2.1014) |
| C12 | 6 / 7 | Sparsity of the 252 cells (2 genders × 7 age groups × 18 genres) | Distinct users per cell: min 11, median 270.5, max 1,538; 41 cells under 100 users | Tier frozen before any satisfaction rate was computed: reliable ≥ 100 distinct users, indicative < 100 | Threshold fixed in D1 Section 5, so it cannot be tuned to the outcome | 252 cells → 211 reliable / 41 indicative |
| C13 | 7b | Cells used for ranking | 211 reliable cells; the 41 indicative cells are listed in Step 6 | Kept reliable cells for ranking; indicative cells set aside and reported as indicative only, not pooled | D1 Section 5 allows “pooled or reported as indicative only”; the second option needs no extra rule | 252 → 211 cells |
| C14 | 8 | Satisfaction rate per cell | satisfied = rating ≥ 4; all rates in [0, 1] | Rate reported without a confidence interval | Ratings from one user are not independent; user-level intervals are computed in D3 | 252 cells with a rate |
| C15 | 9 | Profile of the final working table `df` | 1,000,209 rows × 10 columns, 0 missing, 81 distinct years, 7 age groups | Final | Table handed to the analysts | 10 columns |
| C16 | 2c | User activity | Fewest ratings by one user: 20 | No activity filter | GroupLens already kept only users with at least 20 ratings; heavy raters are handled by counting cell size in distinct users | unchanged |
| C17 | 2d | Additional checks (Data Analyst A, Que) | Row counts match the README (1,000,209 ratings, 6,040 users); 0 ratings dated before the film's release year; 774,028 ratings (77.4%) share the same second with another rating by the same user, largest burst 30 | Keep as is | Timestamp records when a rating was entered, often in bulk, not when the film was watched, so no conclusion in this project relies on rating time | unchanged |
| C18 | B1 | Discovery and validation split | Users of each of the 14 segments split 50/50 with seed 42; discovery 3,018 users (504,116 ratings), validation 3,022 users (496,093 ratings); halves differ by at most one user per segment | Each user is assigned to exactly one half; the extra user of an odd-sized segment goes to validation | Genres are chosen on one half and tested on the other, so selection noise is not tested on the same data; the split is by user because ratings from one user are not independent | 6,040 users → 3,018 discovery / 3,022 validation |
| C19 | 1 | Data source | Colab could not verify the GroupLens certificate on 2026-10-24 (SSL CERTIFICATE_VERIFY_FAILED); in this run Step 1 printed "source : https://files.grouplens.org/datasets/movielens/ml-1m.zip (download, attempt 1)" | Download with certifi certificates, up to 3 attempts; if all fail, read `data/ml-1m.zip` (a local copy of the same official file, never committed) | Certificate verification is never turned off; the data stays the official GroupLens file | unchanged |